# Gabon from space — a visual gallery

**ESA MAAP Hackathon 2026 — Challenge #5 "Canopy Clash"** · companion of notebooks 01 and 02

The analysis notebooks answer *how good* BIOMASS forest height is. This one shows *what* we are measuring: one of the most spectacular forests on Earth, seen by four instruments.

| Figure | What | Data |
|---|---|---|
| 1 | The whole country, forest height on relief | BIOMASS (workshop mosaic) + Copernicus DEM 90 m |
| 2 | Lopé in true colour: savanna, gallery forests and the Ogooué river | Sentinel-2 (cloud-free composite of 17 scenes) |
| 3 | Lopé in 3D: what the eye sees and what the radar measures | Sentinel-2 + BIOMASS L2A on Copernicus DEM 30 m |
| 4 | The airborne LiDAR at full resolution | LVIS RH98, 25 m |

Downloads are cached in data/gallery, so the second run is fast. The figures are saved in results/gallery at print quality, and the last cell builds a slide deck with them.

In [ ]:
# Setup
import sys, json, time, glob, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.colors import LinearSegmentedColormap, LightSource, Normalize, PowerNorm, rgb_to_hsv, hsv_to_rgb
from scipy import ndimage
import rasterio
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling
from rasterio.transform import from_origin
from rasterio.crs import CRS
from rasterio.warp import reproject
warnings.filterwarnings("ignore")

ROOT = next((c for c in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
             if (c / "src" / "__init__.py").exists() and (c / "config" / "sites.json").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from inside the topic folder (it needs src/ and config/).")
sys.path.insert(0, str(ROOT))

CFG = json.loads((ROOT / "config" / "sites.json").read_text()); SITES = CFG["sites"]
OUT = ROOT / "results" / "gallery"; OUT.mkdir(parents=True, exist_ok=True)
CACHE = ROOT / "data" / "gallery"; CACHE.mkdir(parents=True, exist_ok=True)
WS_MOSAIC = Path(CFG["workshop_fh_dir"]).expanduser() / "tif-mosaic" / "mosaic_FH_100m.tif"
WGS84 = CRS.from_epsg(4326)

BG, FG, MUTED = "#070a08", "#eef2ec", "#a9b3a8"
HEIGHT = LinearSegmentedColormap.from_list("height", ["#e9e4c9", "#b5d37a", "#5fae4e", "#1f7a3a", "#0b4a24", "#062b16"])
CANOPY = LinearSegmentedColormap.from_list("canopy", ["#0c1a12", "#16452a", "#2e8b3e", "#9fcf55", "#f2f08a"])
GLOW = [pe.withStroke(linewidth=3, foreground="black")]
FIGS = {}

def save(fig, name, dpi=200):
    p = OUT / f"{name}.png"; fig.savefig(p, dpi=dpi, facecolor=fig.get_facecolor()); FIGS[name] = p; return p

def tile_code(lat, lon):
    return ("N" if lat >= 0 else "S") + f"{abs(lat):02d}", ("E" if lon >= 0 else "W") + f"{abs(lon):03d}"

def to_grid(path, bbox, res, band=1, resampling=Resampling.average):
    """Any raster (local or URL) on a north-up lon/lat grid covering bbox = [W, S, E, N]."""
    W, S, E, N = bbox; w, h = int(round((E - W) / res)), int(round((N - S) / res)); tr = from_origin(W, N, res, res)
    out = np.full((h, w), np.nan, "float32")
    with rasterio.open(path) as src:
        reproject(rasterio.band(src, band), out, src_crs=src.crs or WGS84, dst_transform=tr, dst_crs=WGS84,
                  resampling=resampling, dst_nodata=np.nan)
    return out, tr

def copdem(bbox, res, which=30):
    """Copernicus DEM (30 or 90 m, AWS open data) on the grid of bbox."""
    W, S, E, N = bbox; out = None
    for lat in range(int(np.floor(S)), int(np.ceil(N))):
        for lon in range(int(np.floor(W)), int(np.ceil(E))):
            ns, ew = tile_code(lat, lon); name = f"Copernicus_DSM_COG_{10 if which == 30 else 30}_{ns}_00_{ew}_00_DEM"
            try:
                a, tr = to_grid(f"https://copernicus-dem-{which}m.s3.amazonaws.com/{name}/{name}.tif", bbox, res)
            except Exception:
                continue                       # ocean tiles do not exist
            out = a if out is None else np.where(np.isfinite(a), a, out)
    return out

def enhance(rgb, lo=0.3, hi=99.8, gamma=0.95, sat=1.3):
    """Contrast stretch + a little saturation, for display only."""
    c = np.moveaxis(np.asarray(rgb, float), 0, -1); m = np.isfinite(c).all(-1)
    a, z = np.percentile(c[m], lo), np.percentile(c[m], hi)
    o = np.clip((c - a) / (z - a), 0, 1) ** gamma; o[~m] = 0
    h = rgb_to_hsv(o); h[..., 1] = np.clip(h[..., 1] * sat, 0, 1)
    return hsv_to_rgb(h)

def title_block(fig, x, y, title, subtitle, size=30):
    fig.text(x, y, title, color=FG, fontsize=size, fontweight="bold", va="top", path_effects=GLOW)
    fig.text(x, y - 0.045 * size / 30, subtitle, color=MUTED, fontsize=12, va="top", path_effects=GLOW)

print("Project:", ROOT)

## 1 — Gabon: a country made of forest

Almost 90% of Gabon is covered by forest. Here every pixel is coloured by the canopy height measured by the BIOMASS radar (workshop mosaic, November 2025 – June 2026) and shaded with the Copernicus terrain model, as if lit by the sun from the north-west.
- **Light** = short vegetation: the coastal savannas, and the long diagonal savanna corridors of the Ngounié and Nyanga valleys in the south.
- **Dark green** = tall closed forest, over the Crystal Mountains and the Chaillu massif.
- **White boxes** = our three LiDAR sites.

In [ ]:
GAB = [8.55, -4.15, 14.65, 2.45]                             # W, S, E, N
with rasterio.open(WS_MOSAIC) as src:
    f = 4
    H = src.read(1, out_shape=(src.height // f, src.width // f), resampling=Resampling.average).astype(float); b = src.bounds
    gtr = src.transform * src.transform.scale(src.width / H.shape[1], src.height / H.shape[0])
H[(H < 0) | (H == -9999)] = np.nan
cache = CACHE / "gabon_dem90_on_mosaic_f4.npy"
if cache.exists():
    DEM = np.load(cache)
else:                                                         # first run: ~5 minutes
    DEM = np.full(H.shape, np.nan, "float32")
    for lat in range(int(np.floor(b.bottom)), int(np.ceil(b.top))):
        for lon in range(int(np.floor(b.left)), int(np.ceil(b.right))):
            ns, ew = tile_code(lat, lon); name = f"Copernicus_DSM_COG_30_{ns}_00_{ew}_00_DEM"
            try:
                with rasterio.open(f"https://copernicus-dem-90m.s3.amazonaws.com/{name}/{name}.tif") as s:
                    tmp = np.full(H.shape, np.nan, "float32")
                    reproject(rasterio.band(s, 1), tmp, dst_transform=gtr, dst_crs=WGS84, resampling=Resampling.average, dst_nodata=np.nan)
                    DEM = np.where(np.isfinite(tmp), tmp, DEM)
            except Exception:
                pass
    np.save(cache, DEM)

res_m = abs(gtr.a) * 111320
m = np.isfinite(H); Hs = ndimage.uniform_filter(np.where(m, H, 0), 3) / np.maximum(ndimage.uniform_filter(m.astype(float), 3), 1e-6); Hs[~m] = np.nan
land = np.isfinite(DEM) & (DEM > 0.5)
hs = LightSource(azdeg=315, altdeg=35).hillshade(np.nan_to_num(DEM), vert_exag=10, dx=res_m, dy=res_m)
img = np.clip(HEIGHT(Normalize(0, 50)(np.nan_to_num(Hs)))[..., :3] * (0.22 + 1.0 * hs)[..., None], 0, 1)
img[~np.isfinite(Hs) & land] = np.array([0.16, 0.15, 0.13]) * (0.35 + 0.9 * hs[~np.isfinite(Hs) & land, None])
img[~land] = [0.035, 0.06, 0.09]
c0, c1 = int((GAB[0] - b.left) / gtr.a), int((GAB[2] - b.left) / gtr.a)
r0, r1 = int((b.top - GAB[3]) / -gtr.e), int((b.top - GAB[1]) / -gtr.e)

fig = plt.figure(figsize=(12, 13), facecolor=BG); ax = fig.add_axes([0, 0, 1, 1])
ax.imshow(img[r0:r1, c0:c1], extent=[GAB[0], GAB[2], GAB[1], GAB[3]], interpolation="lanczos"); ax.axis("off")
for k, c in SITES.items():
    w_, s_, e_, n_ = c["bbox"]
    ax.add_patch(plt.Rectangle((w_, s_), e_ - w_, n_ - s_, fill=False, ec="white", lw=1.6))
    ax.text(e_ + 0.07, n_, c["name"], color="white", fontsize=14, va="top", path_effects=GLOW)
title_block(fig, 0.035, 0.975, "GABON", "Forest height seen by the BIOMASS P-band radar\nover Copernicus terrain · Nov 2025 – Jun 2026", size=34)
cax = fig.add_axes([0.06, 0.05, 0.3, 0.013])
cb = fig.colorbar(plt.cm.ScalarMappable(Normalize(0, 50), HEIGHT), cax=cax, orientation="horizontal")
cb.set_label("Canopy height [m]", color=FG); cb.ax.tick_params(colors=FG); cb.outline.set_visible(False)
save(fig, "01_gabon"); plt.show()

## 2 — Lopé in true colour

Lopé National Park (UNESCO World Heritage) is where the forest meets the savanna. In the north the **Ogooué river** cuts through a patch of grassland, and the forest enters the savanna along every stream as dark **gallery forests**: a pattern left by the last Ice Age, when the savanna was much larger and the forest is slowly winning it back.

Gabon is one of the cloudiest places on Earth, so no single Sentinel-2 image is clear. We combine 17 scenes (2018–2024, < 15% cloud each): clouds are masked by brightness and, for every pixel, we keep a low percentile of the remaining dates, which removes haze and cloud remnants.

In [ ]:
LOPE = SITES["lope"]["bbox"]; RES_S2 = 0.00018
cache = CACHE / "lope_s2_composite_u8.npy"
if cache.exists():
    COMP = np.load(cache).astype("float32")
else:                                                         # first run: ~4 minutes
    from pystac_client import Client
    items = list(Client.open("https://earth-search.aws.element84.com/v1").search(
        collections=["sentinel-2-l2a"], bbox=LOPE, query={"eo:cloud_cover": {"lt": 15}}, max_items=60).items())
    W, S, E, N = LOPE; w, h = int(np.ceil((E - W) / RES_S2)), int(np.ceil((N - S) / RES_S2)); tr = from_origin(W, N, RES_S2, RES_S2)
    stack = []
    for x in items:
        with rasterio.open(x.assets["visual"].href) as src, WarpedVRT(src, crs=WGS84, transform=tr, width=w, height=h, resampling=Resampling.average) as v:
            a = v.read().astype("float32")
        a[:, (a.sum(0) == 0) | (a.mean(0) > 120)] = np.inf      # no data or cloud
        stack.append(a)
    st = np.sort(np.stack(stack), axis=0); n = np.isfinite(st).sum(0)
    COMP = np.take_along_axis(st, np.clip((0.3 * (n - 1)).astype(int), 0, None)[None], 0)[0]; COMP[n == 0] = 0
    np.save(cache, COMP.astype("uint8"))
W, S, E, N = LOPE
xcut = 11.78                                                  # east of this line only one tile/date is available (visible seam)
TC = enhance(COMP[:, :, :int((xcut - W) / RES_S2)])
ZOOM = [11.55, -0.20, 11.72, -0.03]

fig = plt.figure(figsize=(18, 13.5), facecolor=BG)
a0 = fig.add_axes([0.0, 0.0, 0.42, 1.0]); a1 = fig.add_axes([0.43, 0.08, 0.57, 0.80])
a0.imshow(TC, extent=[W, xcut, S, N], interpolation="lanczos"); a0.axis("off")
a0.add_patch(plt.Rectangle((ZOOM[0], ZOOM[1]), ZOOM[2] - ZOOM[0], ZOOM[3] - ZOOM[1], fill=False, ec="white", lw=1.5))
r0, r1 = int((N - ZOOM[3]) / RES_S2), int((N - ZOOM[1]) / RES_S2); c0, c1 = int((ZOOM[0] - W) / RES_S2), int((ZOOM[2] - W) / RES_S2)
a1.imshow(TC[r0:r1, c0:c1], extent=[ZOOM[0], ZOOM[2], ZOOM[1], ZOOM[3]], interpolation="lanczos"); a1.axis("off")
a1.text(0.01, 0.02, "Ogooué river · savanna · gallery forests", transform=a1.transAxes, color=FG, fontsize=13, path_effects=GLOW)
title_block(fig, 0.45, 0.975, "LOPÉ", "Where the rainforest meets the savanna · Sentinel-2 true colour, 17 scenes 2018–2024", size=32)
save(fig, "02_lope_true_colour"); plt.show()

## 3 — Lopé in 3D: what the eye sees, what the radar measures

The same landscape draped on the Copernicus 30 m terrain (vertical exaggeration ×4), seen from the south looking towards the Ogooué. Top: Sentinel-2 true colour. Bottom: forest height from the official BIOMASS L2A product (January 2026), with the same colours as Figure 1.

In the radar view the savanna turns pale and the forest dark, but look at the **hills**: on the slopes facing the radar the forest looks lower than it is. That is the foreshortening effect measured in notebooks 01 and 02.

In [ ]:
BOX = [11.39674, -0.40, 11.78, -0.0156]; R3 = 0.0008
cache = CACHE / "lope_dem30_3d.npy"
if cache.exists():
    Z3 = np.load(cache)
else:
    Z3 = copdem(BOX, R3, 30); np.save(cache, Z3)
h3, w3 = Z3.shape
r0, c0 = int((N - BOX[3]) / RES_S2), int((BOX[0] - W) / RES_S2)
tc = TC[r0:r0 + int(round((BOX[3] - BOX[1]) / RES_S2)), c0:c0 + int(round((BOX[2] - BOX[0]) / RES_S2))]
tc3 = np.clip(ndimage.zoom(tc, (h3 / tc.shape[0], w3 / tc.shape[1], 1), order=1), 0, 1)[:h3, :w3]
l2a = None
for p in sorted(glob.glob(str(ROOT / "data" / "lope" / "scenes" / "*20260114*_height.tif"))):
    a, _ = to_grid(p, BOX, R3)
    l2a = a if l2a is None else np.where(np.isfinite(l2a), l2a, a)
hc = HEIGHT(Normalize(0, 50)(np.nan_to_num(l2a)))[..., :3]; hc[~np.isfinite(l2a)] = 0.18

shade = LightSource(azdeg=300, altdeg=30).hillshade(Z3, vert_exag=2, dx=R3 * 111320, dy=R3 * 111320)
LON, LAT = np.meshgrid(np.linspace(BOX[0], BOX[2], w3), np.linspace(BOX[3], BOX[1], h3))
xk, yk = (LON - BOX[0]) * 111.32, (LAT - BOX[1]) * 111.32

def drape(colours, name, title, subtitle):
    col = np.clip(colours * (0.55 + 0.65 * shade[..., None]), 0, 1)
    fig = plt.figure(figsize=(18, 10), facecolor=BG)
    ax = fig.add_axes([-0.25, -0.35, 1.5, 1.7], projection="3d", facecolor=BG)
    ax.plot_surface(xk, yk, Z3, facecolors=col, rstride=1, cstride=1, linewidth=0, antialiased=False, shade=False)
    ax.set_box_aspect((xk.max(), yk.max(), (np.nanmax(Z3) - np.nanmin(Z3)) / 1000 * 4), zoom=1.25)
    ax.view_init(elev=30, azim=-95); ax.set_axis_off()
    title_block(fig, 0.03, 0.96, title, subtitle, size=26)
    save(fig, name, dpi=160); plt.show()

drape(tc3, "03a_lope_3d_true_colour", "LOPÉ IN 3D", "Sentinel-2 true colour on Copernicus terrain (×4) · looking north to the Ogooué")
drape(hc, "03b_lope_3d_biomass", "WHAT THE RADAR SEES", "BIOMASS L2A forest height, January 2026 · light = short, dark = tall")

## 4 — The airborne LiDAR at full resolution

The NASA LVIS survey (AfriSAR, 2016) at its native 25 m pixel: the reference used in all our analyses. At this scale the texture of the forest appears, with emergent trees, gaps, rivers and the sharp forest–savanna boundary. Right: a 6 × 6 km window, chosen automatically where the landscape is most varied (forest–savanna edge, rivers, gaps).

In [ ]:
lvis_path = sorted(p for p in glob.glob(str(ROOT / "data" / "lope" / "*.tif")) if "rh98" in p.lower())
with rasterio.open(lvis_path[0]) as src:
    LV = src.read(1).astype(float); lb = src.bounds; LV[LV == src.nodata] = np.nan; LV[LV < 0] = np.nan
k = 240                                                       # 240 px = 6 km
# window with full coverage and the most varied landscape (forest-savanna edge)
v = np.isfinite(LV).astype(float); L0 = np.nan_to_num(LV)
cnt = ndimage.uniform_filter(v, k); m1 = ndimage.uniform_filter(L0, k) / np.maximum(cnt, 1e-6)
sd = np.sqrt(np.maximum(ndimage.uniform_filter(L0 ** 2, k) / np.maximum(cnt, 1e-6) - m1 ** 2, 0))
score = np.where(cnt > 0.95, sd, -1)
rc, cc = np.unravel_index(np.argmax(score), score.shape)
r0, c0 = int(np.clip(rc - k // 2, 0, LV.shape[0] - k)), int(np.clip(cc - k // 2, 0, LV.shape[1] - k))
fig = plt.figure(figsize=(18, 10.5), facecolor=BG)
a0 = fig.add_axes([0.0, 0.02, 0.45, 0.86], facecolor=BG); a1 = fig.add_axes([0.47, 0.1, 0.5, 0.78], facecolor=BG)
ext = [lb.left / 1000, lb.right / 1000, lb.bottom / 1000, lb.top / 1000]
a0.imshow(LV, extent=ext, cmap=CANOPY, norm=Normalize(0, 55), interpolation="nearest"); a0.axis("off")
x0, y0 = lb.left + c0 * 25, lb.top - (r0 + k) * 25
a0.add_patch(plt.Rectangle((x0 / 1000, y0 / 1000), k * 25 / 1000, k * 25 / 1000, fill=False, ec="white", lw=1.5))
im = a1.imshow(LV[r0:r0 + k, c0:c0 + k], cmap=CANOPY, norm=Normalize(0, 55), interpolation="nearest"); a1.axis("off")
cax = fig.add_axes([0.6, 0.05, 0.25, 0.013]); cb = fig.colorbar(im, cax=cax, orientation="horizontal")
cb.set_label("LVIS RH98 canopy height [m]", color=FG); cb.ax.tick_params(colors=FG); cb.outline.set_visible(False)
title_block(fig, 0.02, 0.975, "EVERY TREE COUNTS", "NASA LVIS airborne LiDAR over Lopé, 25 m pixels (AfriSAR 2016)", size=26)
save(fig, "04_lvis_25m"); plt.show()

## 5 — Slide deck

Full-screen slides with the five figures: results/Gabon_gallery.pptx. They can also be copied into the analysis decks of notebooks 01 and 02.

In [ ]:
from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.dml.color import RGBColor
from PIL import Image
prs = Presentation(); prs.slide_width, prs.slide_height = Inches(13.333), Inches(7.5)
for name in ["01_gabon", "02_lope_true_colour", "03a_lope_3d_true_colour", "03b_lope_3d_biomass", "04_lvis_25m"]:
    if name not in FIGS: continue
    s = prs.slides.add_slide(prs.slide_layouts[6])
    bg = s.shapes.add_shape(1, 0, 0, prs.slide_width, prs.slide_height); bg.fill.solid(); bg.fill.fore_color.rgb = RGBColor(7, 10, 8); bg.line.fill.background()
    w, h = Image.open(FIGS[name]).size; sc = min(13.333 / w, 7.5 / h)
    s.shapes.add_picture(str(FIGS[name]), Inches((13.333 - w * sc) / 2), Inches((7.5 - h * sc) / 2), Inches(w * sc), Inches(h * sc))
report = ROOT / "results" / "Gabon_gallery.pptx"; prs.save(report); print("Deck:", report, f"({len(prs.slides)} slides)")